In [ ]:
# SETUP
from google.colab import drive
drive.mount('/content/drive')

!pip install mne --quiet
import os
import mne
import matplotlib.pyplot as plt
import gc
import traceback

In [ ]:
import os
import mne
import matplotlib.pyplot as plt
import gc
import traceback

# Root EEG directory
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'

# Target subjects
target_subjects = ['3', '6', '7', '8', '10']

# Filter participants list to include only target subjects
participants = sorted([
    p for p in os.listdir(root_dir)
    if os.path.isdir(os.path.join(root_dir, p)) and p in target_subjects
])

def run_ica_excluding_breaks(participant):
    participant_path = os.path.join(root_dir, participant)
    fif_path = os.path.join(participant_path, f'{participant}_filtered_raw.fif')
    ica_path = os.path.join(participant_path, f'{participant}_ica.fif')

    if not os.path.exists(fif_path):
        print(f"[{participant}] ❌ No filtered .fif file found.")
        return
    if os.path.exists(ica_path):
        print(f"[{participant}] ✅ ICA already exists, skipping.")
        return

    print(f"\n>>> Running ICA for {participant} ")

    try:
        raw = mne.io.read_raw_fif(fif_path, preload=True)
        raw.set_montage('GSN-HydroCel-128')

        # Crop to 10 minutes max to reduce memory load
        raw.crop(tmax=min(600, raw.times[-1]))

        # Mark 'break' annotations as bad segments
        if raw.annotations:
            break_indices = [i for i, desc in enumerate(raw.annotations.description) if desc.lower() == 'break']
            if break_indices:
                print(f"[{participant}] Found {len(break_indices)} 'break' annotations — excluding them from ICA.")
                raw.annotations.delete(break_indices)

        # Pick EEG channels only
        picks = mne.pick_types(raw.info, eeg=True, exclude='bads')

        # Run ICA
        ica = mne.preprocessing.ICA(n_components=None, method='fastica', random_state=97, max_iter=500)
        ica.fit(raw, picks=picks, reject_by_annotation=True)
        print(f"[{participant}] ✓ ICA fitted excluding breaks.")

        # Save ICA solution
        ica.save(ica_path, overwrite=True)
        print(f"[{participant}] ✓ ICA saved: {ica_path}")

        # Save ICA topomap in a single figure
        fig_list = ica.plot_components(inst=raw, show=False)
        fig = fig_list[0] if isinstance(fig_list, list) else fig_list
        fig_comp_path = os.path.join(participant_path, f"{participant}_ica_components.png")
        fig.savefig(fig_comp_path, dpi=150)
        print(f"[{participant}] ✓ ICA component map saved.")

        # Save ICA time series
        fig_sources = ica.plot_sources(raw, show=False)
        fig_src_path = os.path.join(participant_path, f"{participant}_ica_sources.png")
        fig_sources.savefig(fig_src_path, dpi=150)
        print(f"[{participant}] ✓ ICA time series plot saved.")

        del raw, ica, fig, fig_sources
        gc.collect()

    except Exception as e:
        print(f"[{participant}] ⚠️ ERROR: {e}")
        traceback.print_exc()

# === Run for target subjects only ===
for pid in participants:
    run_ica_excluding_breaks(pid)


In [ ]:
# Paths
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'
target_subjects = ['10', '11']

for subject in target_subjects:
    print(f"\n=== Subject {subject} ===")
    participant_path = os.path.join(root_dir, subject)
    raw_path = os.path.join(participant_path, f'{subject}_filtered_raw.fif')
    ica_path = os.path.join(participant_path, f'{subject}_ica.fif')

    # Load data
    raw = mne.io.read_raw_fif(raw_path, preload=True)
    ica = mne.preprocessing.read_ica(ica_path)
    raw.set_montage('GSN-HydroCel-128')

    # Plot ICA components (topomap)
    fig_topos = ica.plot_components(inst=raw, show=False)
    for i, fig in enumerate(fig_topos):
        fig_path = os.path.join(participant_path, f"{subject}_ica_topomap_panel_{i+1}.png")
        fig.savefig(fig_path, dpi=150)
        plt.close(fig)
        print(f"Saved topomap panel {i+1} for Subject {subject} to {fig_path}")

    # Plot ICA time series
    fig_sources = ica.plot_sources(raw, show=False)
    sources_path = os.path.join(participant_path, f"{subject}_ica_sources.png")
    fig_sources.savefig(sources_path, dpi=150)
    plt.close(fig_sources)
    print(f"Saved time series plot for Subject {subject} to {sources_path}")

In [ ]:
import os
import mne
import matplotlib.pyplot as plt
import gc
import traceback

# Root EEG directory
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'
participants = sorted([p for p in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, p))])

def run_ica_excluding_breaks(participant):
    participant_path = os.path.join(root_dir, participant)
    fif_path = os.path.join(participant_path, f'{participant}_filtered_raw.fif')
    ica_path = os.path.join(participant_path, f'{participant}_ica.fif')

    if not os.path.exists(fif_path):
        print(f"[{participant}] ❌ No filtered .fif file found.")
        return
    if os.path.exists(ica_path):
        print(f"[{participant}] ✅ ICA already exists, skipping.")
        return

    print(f"\n>>> Running ICA for {participant} ")

    try:
        raw = mne.io.read_raw_fif(fif_path, preload=True)
        raw.set_montage('GSN-HydroCel-128')

        # Crop to 10 minutes max to reduce memory load
        raw.crop(tmax=min(600, raw.times[-1]))

        # Mark 'break' annotations as bad segments
        if raw.annotations:
            break_indices = [i for i, desc in enumerate(raw.annotations.description) if desc.lower() == 'break']
            if break_indices:
                print(f"[{participant}] Found {len(break_indices)} 'break' annotations — excluding them from ICA.")
                raw.annotations.delete(break_indices)

        # Pick EEG channels only
        picks = mne.pick_types(raw.info, eeg=True, exclude='bads')

        # Run ICA
        ica = mne.preprocessing.ICA(n_components=None, method='fastica', random_state=97, max_iter=500)
        ica.fit(raw, picks=picks, reject_by_annotation=True)
        print(f"[{participant}] ✓ ICA fitted excluding breaks.")

        # Save ICA solution
        ica.save(ica_path, overwrite=True)
        print(f"[{participant}] ✓ ICA saved: {ica_path}")

        # Save ICA topomap in a single figure
        fig_list = ica.plot_components(inst=raw, show=False)
        fig = fig_list[0] if isinstance(fig_list, list) else fig_list
        fig_comp_path = os.path.join(participant_path, f"{participant}_ica_components.png")
        fig.savefig(fig_comp_path, dpi=150)
        print(f"[{participant}] ✓ ICA component map saved.")

        # Save ICA time series
        fig_sources = ica.plot_sources(raw, show=False)
        fig_src_path = os.path.join(participant_path, f"{participant}_ica_sources.png")
        fig_sources.savefig(fig_src_path, dpi=150)
        print(f"[{participant}] ✓ ICA time series plot saved.")

        del raw, ica, fig, fig_sources
        gc.collect()

    except Exception as e:
        print(f"[{participant}] ⚠️ ERROR: {e}")
        traceback.print_exc()

# === Run for all participants ===
for pid in participants:
    run_ica_excluding_breaks(pid)


In [ ]:
import os
import mne
import matplotlib.pyplot as plt

# Paths
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'
target_subjects = ['10', '11']

for subject in target_subjects:
    print(f"\n=== Subject {subject} ===")
    participant_path = os.path.join(root_dir, subject)
    raw_path = os.path.join(participant_path, f'{subject}_filtered_raw.fif')
    ica_path = os.path.join(participant_path, f'{subject}_ica.fif')

    # Load data
    raw = mne.io.read_raw_fif(raw_path, preload=True)
    ica = mne.preprocessing.read_ica(ica_path)
    raw.set_montage('GSN-HydroCel-128')

    # Plot ICA components (topomap)
    fig_topos = ica.plot_components(inst=raw, show=False)
    for i, fig in enumerate(fig_topos):
        fig_path = os.path.join(participant_path, f"{subject}_ica_topomap_panel_{i+1}.png")
        fig.savefig(fig_path, dpi=150)
        plt.close(fig)
        print(f"Saved topomap panel {i+1} for Subject {subject} to {fig_path}")

    # Plot ICA time series
    fig_sources = ica.plot_sources(raw, show=False)
    sources_path = os.path.join(participant_path, f"{subject}_ica_sources.png")
    fig_sources.savefig(sources_path, dpi=150)
    plt.close(fig_sources)
    print(f"Saved time series plot for Subject {subject} to {sources_path}")
